# Exp118 - ILP cost grid on the MINIMAL branch

Diagnostic. **Writes no submission and costs no submission slot.**

## Why this is not a repeat of Exp110-115

Exp110-115 swept `ILP_DISAPPEARANCE_WEIGHT` (1.2 -> 1.8) and the public LB was
flat (`0.908` / `0.909` / `0.909` / `0.909` / `0.909`). That sweep was
**measuring almost nothing**: on the Exp073/Exp110 branch, `filter_output_graph`
step 4 (`edges = motion_edges`) discarded *all* ~114,860 ILP edges and rebuilt
association with a greedy Hungarian sweep. The ILP only chose which nodes
survived, never which edges were submitted.

On the minimal branch the ILP output **is** the submission. So these weights now
determine the graph directly, and `1.4` - inherited from the same lineage via
`hengck23/minimal-baseline-tta-2gpu` and never independently validated - is worth
re-testing.

Exp117 already settled `ILP_DIVISION_WEIGHT`: keep it at `1.0` (lower values
produce forks but zero true positives and cost edge quality). It is held fixed
here.

## Grid

- `ILP_DISAPPEARANCE_WEIGHT` x `ILP_APPEARANCE_WEIGHT`, plus two
  `ILP_EDGE_WEIGHT` probes at the inherited operating point.
- Inference is independent of ILP costs, so it runs ONCE per movie and every
  configuration re-solves only the ILP on the cached candidate graph.
- Every configuration is scored with the OFFICIAL metric.

## Reading the output

Per `LEARNINGS.md`: the local aggregate under-reads the public LB by ~`0.035`,
and the three `44b6` movies carry only ~50 GT edges each and score perfectly, so
they cannot discriminate. **Judge configurations on the two `6bba` movies**,
which carry ~94% of the local weight - especially `6bba_05db0fb1`, which holds
essentially all measurable loss (79 FP / 88 FN).


In [ ]:
try:
    import zarr, geff, tracksdata
    
except: 
    import os
    import sys
    import subprocess
    import importlib
    import importlib.util
    from pathlib import Path
    
    # Polars wheel in the support package uses the 32-bit runtime package.
    os.environ.setdefault("POLARS_PREFER_PKG", "32")
    
    SUPPORT_DIR = Path(
        "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1"
    )
    WHEELS_DIR = SUPPORT_DIR / "wheels"
    
    if not WHEELS_DIR.exists():
        # Fallback for Kaggle's shorter mounted dataset path.
        candidates = list(Path("/kaggle/input").glob("**/wheels"))
        if not candidates:
            raise FileNotFoundError("Could not find the attached offline wheels directory")
        WHEELS_DIR = candidates[0]
    
    print("Offline wheels:", WHEELS_DIR)
    
    
    # These are packages supplied by the attached dataset.
    # Deliberately exclude:
    #   numpy, scipy, torch, pandas, dask, xarray, numba, llvmlite
    #
    # Kaggle already supplies compatible versions of those packages.
    OFFLINE_PACKAGES = [
        "tracksdata",
        "zarr==3.2.1",
        "numcodecs==0.15.1",
        "donfig==0.8.1.post1",
        "geff==1.2.0.1.1",
        "geff-spec==1.1.1",
        "pyscipopt==6.2.1",
        "ilpy==0.6.0",
        "rustworkx==0.18.0",
        "polars==1.42.0",
        "polars-runtime-32==1.42.0",
        "bidict==0.23.1",
        "imagecodecs==2026.6.26",
    ]
    
    
    def module_missing(module_name: str) -> bool:
        return importlib.util.find_spec(module_name) is None
    
    
    # Import name differs from pip package name in several cases.
    REQUIRED_IMPORTS = {
        "tracksdata": "tracksdata",
        "zarr": "zarr",
        "numcodecs": "numcodecs",
        "geff": "geff",
        "pyscipopt": "pyscipopt",
        "ilpy": "ilpy",
        "rustworkx": "rustworkx",
        "polars": "polars",
        "imagecodecs": "imagecodecs",
    }
    
    
    def purge_modules(module_roots):
        """
        Remove already-imported package modules from sys.modules.
    
        Normally this cell runs before imports, but this also protects against
        accidental imports performed by earlier Kaggle initialization code.
        """
        for root in module_roots:
            for name in list(sys.modules):
                if name == root or name.startswith(root + "."):
                    sys.modules.pop(name, None)
    
    
    def install_offline_packages():
        cmd = [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-index",
            "--no-deps",
            "--find-links",
            str(WHEELS_DIR),
            *OFFLINE_PACKAGES,
        ]
    
        print("Installing attached packages without modifying NumPy/SciPy/Torch...")
        result = subprocess.run(
            cmd,
            text=True,
            capture_output=True,
        )
    
        if result.returncode != 0:
            print(result.stdout[-4000:])
            print(result.stderr[-4000:])
            raise RuntimeError("Offline dependency installation failed")
    
        purge_modules(REQUIRED_IMPORTS.values())
    
    
    install_offline_packages()
    
    
    # Verify the complete import chain needed by biohub_tracking.io.
    failures = {}
    
    for name, module_name in {
        **REQUIRED_IMPORTS,
        "numpy": "numpy",
        "scipy": "scipy",
        "dask": "dask.array",
        "xarray": "xarray",
    }.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    
    if failures:
        raise ImportError(
            "Dependency verification failed:\n"
            + "\n".join(f"{name}: {error}" for name, error in failures.items())
        )

import zarr, geff, tracksdata
print("All offline dependencies imported successfully.")

In [ ]:
#simplified inference pipeline: you can now easily modify to run parallel process for 2 GPU

import sys
sys.path.append("/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/repo/src")
#import biohub_tracking as tracking_cellmot

from biohub_tracking.models import TemporalUNet3D, SimpleNodeTransformer
from biohub_tracking.io import open_dataset, save_graph

import os
import contextlib
import zarr
import numpy as np
from tqdm import tqdm
import json
import glob
import csv
import pandas as pd
from joblib import Parallel, delayed

import torch
import torch.nn as nn
import torch.nn.functional as F

#graph
import tracksdata as td
import polars as pl
import pandas as pd

#evalute
from geff import GeffMetadata
from biohub_tracking.metrics import (
    evaluate,
    node_recall,
    per_sample_metrics,
    summarise,
)

#--------------------------------------------
MODE ="local"  # DIAGNOSTIC: labelled train movies so we can score officially

KAGGLE_DIR = "/kaggle/input/competitions/biohub-cell-tracking-during-development"
if MODE =="local":
    valid_id  = [ '44b6_0113de3b', '44b6_0b24845f', '6bba_05b6850b', '6bba_05db0fb1', '44b6_33b596bf',]
    valid_dir = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train"

if MODE =="submit":
    glob_file = glob.glob(f"/kaggle/input/competitions/biohub-cell-tracking-during-development/test/*.zarr")
    valid_id  = sorted([f.split("/")[-1][:-5] for f in glob_file])
    valid_dir = "/kaggle/input/competitions/biohub-cell-tracking-during-development/test"


print("MODE:", MODE)
print("valid_id:", len(valid_id), valid_id[:4])

print("setup ok!!!!!")

# ---- sweep configuration -------------------------------------------------
# Both directions, because the cost sign convention in tracksdata's ILPSolver
# is not verified. 1.0 is the inherited baseline that produced zero divisions.
# Exp117 settled division weight: 1.0 is optimal. Held fixed.
ILP_DIVISION_WEIGHT_FIXED = 1.0

# (appearance, disappearance, edge_weight); edge_weight None -> keep ILP_EDGE_WEIGHT (-1.0)
COST_GRID = [
    (0.0, 1.0, None),
    (0.0, 1.4, None),   # inherited operating point / hengck23 default
    (0.0, 1.8, None),
    (0.0, 2.2, None),
    (0.5, 1.4, None),
    (1.0, 1.4, None),
    (0.5, 1.8, None),
    (1.0, 1.8, None),
    (0.0, 1.4, -0.5),
    (0.0, 1.4, -2.0),
]

CACHE_DIR = "/kaggle/working/pred_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# Only keep movies that actually have ground truth available locally.
_avail = []
for _sid in valid_id:
    _z = f"{KAGGLE_DIR}/train/{_sid}.zarr"
    _g = f"{KAGGLE_DIR}/train/{_sid}.geff"
    if os.path.exists(_z) and os.path.exists(_g):
        _avail.append(_sid)
    else:
        print(f"SKIP {_sid}: zarr={os.path.exists(_z)} geff={os.path.exists(_g)}")
valid_id = _avail
print("scoring movies:", len(valid_id), valid_id)
assert valid_id, "no labelled movies available - cannot score"


In [ ]:
#modeling

DEVICE = "cuda"
SUBSAMPLE    = [1,4,4]
VOLUME_SHAPE = [64,64,64]
TIME_LENGTH  = 2

POINT_THRESHOLD = 0.9700
USE_TTA = True
USE_MULTI_GPU=True

ILP_EDGE_WEIGHT          = -1.0
ILP_APPEARANCE_WEIGHT    =  0.0
ILP_DISAPPEARANCE_WEIGHT =  1.4
ILP_DIVISION_WEIGHT      =  1.0


class MyUnet(nn.Module):
    def __init__(
        self,
        config
    ):
        super().__init__()
        self.D =nn.Parameter(torch.ones(1))

        self.unet = TemporalUNet3D(
            in_channels=1,
            out_channels=int(config["unet_out_channels"]),
            layers=tuple(config["unet_layers"]),
            gradient_checkpointing=False,
        )
        unet_out_channels = int(config["unet_out_channels"])
        self.unet_out_channels = unet_out_channels
        self.detect_head = nn.Conv3d(unet_out_channels, 1, kernel_size=1)

        pos_feat_dim = 4 * 8
        self.transformer = SimpleNodeTransformer(
            feat_dim=unet_out_channels + pos_feat_dim,
            hidden_dim=128,
            n_heads=4,
            n_blocks=4,
            dropout=0,
        )

    def forward_unet(
        self,
        image: torch.Tensor,  # B,T,Z,Y,X #T=2
    ) -> tuple[torch.Tensor, list[torch.Tensor]]:

        image = image[:,:,None]  # B,T,1,Z,Y,X
        f = self.unet(image)     # B,T,C,Z,Y,X

        point_logit = [
            self.detect_head(f[:, 0]), #t=1,2
            self.detect_head(f[:, 1]),
        ]
        point_feature =[
            f[:, 0],
            f[:, 1],
        ]
        return point_feature, point_logit

    def forward_transformer(
        self,
        select0: torch.Tensor,   # (N0, C) pre-indexed
        select1: torch.Tensor,   # (N1, C)
        coord0: torch.Tensor,    # (N0, 3)
        coord1: torch.Tensor,    # (N1, 3)
        pos0: torch.Tensor,      # (N0, dim)
        pos1: torch.Tensor,      # (N1, dim)

    ) -> torch.Tensor:

        feature0 = torch.cat([select0, pos0], dim=-1)
        feature1 = torch.cat([select1, pos1], dim=-1)
        logit =  self.transformer(
            feature0,
            feature1,
            coord0,
            coord1,
        )
        return logit

## modeling helper -----------------------------------------------------

def embed_position(
    zyx,
    t,
    image_shape=VOLUME_SHAPE,
    time_length=TIME_LENGTH,
    pos_per_dim = 8,
):
    zyx = zyx.float()
    z, y, x = zyx.unbind(dim=1)
    t_tensor = torch.as_tensor(
        t,
        dtype=zyx.dtype,
        device=zyx.device,
    )
    t_normalized = torch.ones_like(z) * (t_tensor / time_length) #todo: t should be 0,1
    tzyx = [
        t_normalized,
        z / image_shape[0],
        y / image_shape[1],
        x / image_shape[2],
    ]

    def embed(values: torch.Tensor) -> torch.Tensor:
        freqs = 2.0 ** torch.arange(
            pos_per_dim // 2,
            dtype=values.dtype,
            device=values.device,
        )
        angles = values[:, None] * freqs[None, :] * torch.pi
        return torch.cat(
            [torch.sin(angles), torch.cos(angles)],
            dim=1,
        )
    return torch.cat([embed(values) for values in tzyx], dim=1)

def pool_kernel_from_um(
    um: float,
    voxel_size: tuple[float, ...],
) -> tuple[int, ...]:
    kernel = []
    for s in voxel_size:
        k = max(1, round(um / s))
        if k % 2 == 0:
            k += 1
        kernel.append(k)
    return tuple(kernel)

def prob_to_zyx(
    prob: torch.Tensor,
    threshold: float = 0.5,
    pool_kernel: tuple[int, ...] = (3, 3, 3),
) -> np.ndarray:

    prob = prob.unsqueeze(0)
    pad = tuple(k // 2 for k in pool_kernel)
    pooled = F.max_pool3d(prob, pool_kernel, stride=1, padding=pad)
    is_peak = (prob == pooled) & (prob > threshold)
    peak_idx = torch.nonzero(is_peak[0, 0])
    if peak_idx.shape[0] == 0:
        return torch.empty((0, 3), dtype=torch.long)
    zyx  =  peak_idx
    return zyx

#todo? interpolation????
def select_feature(
    feature: torch.Tensor,  # (C, Z, Y, X)
    zyx: torch.Tensor,      # (N, 3), coordinates in ZYX order
) -> torch.Tensor:
    _, Z, Y, X = feature.shape
    z = zyx[:, 0].long().clamp(0, Z - 1)
    y = zyx[:, 1].long().clamp(0, Y - 1)
    x = zyx[:, 2].long().clamp(0, X - 1)
    selected = feature[:, z, y, x]
    return selected.permute(1, 0).contiguous()

# Graph building
def build_graph(
    coord,
    edge
):
    graph = td.graph.InMemoryGraph()
    for key in ["z", "y", "x"]:
        graph.add_node_attr_key(key, pl.Float64, -999999.0)

    node_ids = graph.bulk_add_nodes([
        {"t": int(t), "z": float(z), "y": float(y), "x": float(x)}
        for t, z, y, x in coord
    ])

    if edge:
        graph.add_edge_attr_key("edge_prob", pl.Float64, 0.0)
        graph.add_edge_attr_key("edge_dist", pl.Float64, 0.0)
        graph.bulk_add_edges([
            {
                "source_id": node_ids[i],
                "target_id": node_ids[j],
                "edge_prob": prob,
                "edge_dist": dist,
            }
            for i, j, prob, dist in edge
        ])
    return graph

# io helper ----------------------------------

def load_model_weight(weight_file, model):
    state = torch.load( weight_file, map_location="cpu", weights_only=True)
    #state = state["model_state_dict"]
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"loaded weight: {weight_file}")
    print(f"\tmissing key: {len(missing)}", missing)
    print(f"\tunexpected key: {len(unexpected)}", unexpected)
    return model

def load_volume(sample_id):
    zarr_file = f"{valid_dir}/{sample_id}.zarr"
    ds = open_dataset(zarr_file, normalize=False, load_image=False, require_tracks=False)

    zarr_arr = zarr.open_group(str(ds.zarr_path), mode="r")["0"]
    q_low    = float(ds.quantiles["0.001"])
    q_high   = float(ds.quantiles["0.999"])
    dz, dy, dx = SUBSAMPLE
    small = zarr_arr[:, ::dz, ::dy, ::dx].astype(np.float32)
    assert small.shape[1:] == tuple(VOLUME_SHAPE)

    small = ((small - q_low) / (q_high - q_low + 1e-6))
    small = np.clip(small, 0.0, None ) #1.0 :todo  None --> 1.0 is better
    voxel_size = tuple(s * d for s, d in zip(ds.scale, SUBSAMPLE))
    meta = {
        "voxel_size": voxel_size,
    }
    return small, meta


def do_tta_4flip(im):
    image = [im]
    image+= [im.flip(dims=(2,))] # Y
    image+= [im.flip(dims=(3,))] # X
    image+= [im.flip(dims=(2,3))] #XY
    return image, None

def undo_tta_4flip(
    x,
    transform=None,
):
    x[0] = x[0]
    x[1] = x[1].flip(dims=(2,)) # undo Y
    x[2] = x[2].flip(dims=(3,))
    x[3] = x[3].flip(dims=(2,3))
    return x


#---
def do_tta_8yx(im):
    image = []
    transform = []
    for flip_x in (False, True):
        for k in range(4):
            x = im
            # One reflection combined with four rotations gives
            # all 8 distinct square symmetries.
            if flip_x:
                x = x.flip(dims=(-1,))
            x = torch.rot90(x, k=k, dims=(-2, -1))
            image.append(x)
            transform.append((k, flip_x))
    return image, transform

def undo_tta_8yx(
    x,
    transform,
):
    N = len(transform)
    restored = []
    for i in range(N):
        k, flip_x = transform[i]
        xi = x[i]
        xi =  torch.rot90(xi, k=(-k) % 4, dims=(-2, -1))
        if flip_x:
            xi = xi.flip(dims=(-1,))
        restored.append(xi)
    return torch.stack(restored, dim=0)



def do_tta_8fliprot(im):
    dims = (-2, -1)

    images = [
        im,                                      # identity
        im.flip(dims=(-1,)),                    # X flip
        im.flip(dims=(-2,)),                    # Y flip
        im.flip(dims=(-2, -1)),                 # 180° rotation
        torch.rot90(im, 1, dims=dims),          # 90°
        torch.rot90(im, 3, dims=dims),          # 270°
        im.transpose(-1, -2),                   # main-diagonal reflection
        torch.rot90(im, 1, dims=dims)
             .transpose(-1, -2),                # anti-diagonal reflection
    ]
    return images, None


def undo_tta_8fliprot(x, transform=None):
    dims = (-2, -1)

    return torch.stack([
        x[0],
        x[1].flip(dims=(-1,)),
        x[2].flip(dims=(-2,)),
        x[3].flip(dims=(-2, -1)),
        torch.rot90(x[4], -1, dims=dims),
        torch.rot90(x[5], -3, dims=dims),
        x[6].transpose(-1, -2),
        torch.rot90(
            x[7].transpose(-1, -2),
            -1,
            dims=dims,
        ),
    ])



def do_tta_9public(im):
    dims = (-2, -1)

    images = [
        im,                                      # identity
        im.flip(dims=(-1,)),                    # X flip
        im.flip(dims=(-2,)),                    # Y flip
        im.flip(dims=(-2, -1)),                 # XY flip, 180° rotation
        im.rot90(1, dims=dims),          # 90°
        im.rot90(2, dims=dims),          # 180°
        im.rot90(3, dims=dims),          # 270°
        im.transpose(-1, -2),                   # main-diagonal reflection
        im.rot90(1, dims=dims).transpose(-1, -2),  # anti-diagonal reflection
    ]
    return images, None


def undo_tta_9public(x, transform=None):
    dims = (-2, -1)

    return torch.stack([
        x[0],
        x[1].flip(dims=(-1,)),
        x[2].flip(dims=(-2,)),
        x[3].flip(dims=(-2, -1)),
        x[4].rot90(-1, dims=dims),
        x[5].rot90(-2, dims=dims),
        x[6].rot90(-3, dims=dims),
        x[7].transpose(-1, -2),
        x[8].transpose(-1, -2).rot90(-1,dims=dims),
    ])


################################################################

@contextlib.contextmanager
def suppress_output():
    """Context manager to suppress stdout and stderr."""
    with open(os.devnull, "w") as devnull:
        with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
            yield


def predict_one(model, volume, meta):

    point_threshold = POINT_THRESHOLD # 0.5
    pool_kernel_um  = 3.0
    edge_threshold  = 0.5

    # --
    device = model.D.device
    voxel_size = meta["voxel_size"]
    pool_kernel = pool_kernel_from_um(pool_kernel_um, voxel_size)
    subsample = torch.as_tensor([SUBSAMPLE], dtype=torch.float32, device=device)
    T = volume.shape[0]

    #output for graph in submission df
    out_edge  = []
    out_node  = []
    out_start = {}

    # start of out helper ----
    def add_to_out(edge_prob, coord0, coord1, t0, t1):

        if t0==0:
            out_start[t0] = len(out_node)
            for z,y,x in coord0:
                out_node.append([t0, z,y,x])

        out_start[t1] = len(out_node)
        for z, y, x in coord1:
            out_node.append([t1, z, y, x])

        N0,N1 = edge_prob.shape
        candidate = sorted(
            [
                (edge_prob[i, j], i, j)
                for i in range(N0)
                for j in range(N1)
                if edge_prob[i, j] > edge_threshold
            ],
            reverse=True,
        )
        start0 = out_start[t0]
        start1 = out_start[t1]
        for prob, i, j in candidate:
            dist = np.linalg.norm( coord0[i] - coord1[j] )
            out_edge.append([i+start0, j+start1, float(prob), float(dist)])


    # end of out helper ----

    #USE_TTA =False
    for t in tqdm( range(T - 1), total=T - 1, leave=False, disable=False):
        im = torch.from_numpy(volume[t:t+2]).to(device)
        image = [im] #TZYX

        #with torch.no_grad():
        with torch.inference_mode():
            if USE_TTA:
                do_tta, undo_tta = do_tta_8fliprot, undo_tta_8fliprot
                image,transform  = do_tta(im)  # do_tta_4flip(im) 
                

            A = len(image) #num_augment
            image = torch.stack(image, dim=0)
            point_feature, point_logit = model.forward_unet(image)

            if USE_TTA:  # tta
                #C,ZYX #undo_tta_4flip
                point_feature = [ undo_tta(x, transform) for x in  point_feature] # for t=0,1
                point_logit   = [ undo_tta(x, transform) for x in  point_logit]

            #emsemble dilemma : sigmoid(ave(logit)) or ave(sigmoid(logit)))
            point_prob = [torch.sigmoid(x.mean(0)) for x in point_logit]
            #point_prob = [torch.sigmoid(x).mean(0) for x in point_logit]
            zz=0
            # ------------------------------------------------------------------
            if t==0:
                zyx0 = prob_to_zyx(point_prob[0], pool_kernel=pool_kernel, threshold=point_threshold) #(N,3)
            else:
                zyx0 = zyx1  #keep coord from previous

            pos0    = embed_position(zyx0, t=0, pos_per_dim=8)   #392, 32 #t=0 beecuase relative time is used
            coord0  = zyx0 * subsample
            select0 =torch.stack( [
                select_feature(f, zyx0) for f in point_feature[0][:1]
            ])  #392, 32

            zyx1    = prob_to_zyx(point_prob[1], pool_kernel=pool_kernel, threshold=point_threshold)
            pos1    = embed_position(zyx1, t=1, pos_per_dim=8)
            coord1  = zyx1*subsample
            select1 =torch.stack( [
                select_feature(f, zyx1 ) for f in point_feature[1][:1]
            ])
   
            #print(coord1.shape)
            E = len(select0)
            edge_logit = model.forward_transformer(
                select0,
                select1,
                coord0[None].expand(E,-1, -1),
                coord1[None].expand(E,-1, -1),
                pos0[None].expand(E,-1, -1),
                pos1[None].expand(E,-1, -1),
            ) # (N0, N1)
            #edge_prob = torch.softmax(edge_logit, dim=1).mean(0) #same tta has very large value
            edge_prob = torch.softmax(edge_logit.mean(0), dim=0)

            #----------------------------------------
            add_to_out(
                edge_prob.float().data.cpu().numpy(),
                coord0.float().data.cpu().numpy(),
                coord1.float().data.cpu().numpy(),
                t0=t, t1=t+1,
            ) 
            #todo: check correct even if there is empty detection at time t


    return out_node, out_edge



print("modeling ok !!!")

In [ ]:
# STAGE A - run detection/edge inference ONCE per movie and cache the candidate graph.
# Sequential single-GPU: simpler and safer than the 2-GPU split, and only ~2 min/movie.
import time, numpy as np

_n_gpu = torch.cuda.device_count()
print("visible CUDA devices:", _n_gpu)
if _n_gpu < 1:
    raise RuntimeError("No CUDA device visible; this kernel requires a GPU accelerator.")

checkpoint_dir = "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/weights/unet_transformer/split_0"
checkpoint_file = f"{checkpoint_dir}/edge_predictor_best.pth"
config_file     = f"{checkpoint_dir}/config.json"

device = torch.device("cuda:0")
with open(config_file, "r", encoding="utf-8") as f:
    config = json.load(f)
model = MyUnet(config)
load_model_weight(checkpoint_file, model)
model.to(device)
model.eval()

for sample_id in valid_id:
    cache_file = f"{CACHE_DIR}/{sample_id}.npz"
    if os.path.exists(cache_file):
        print(f"{sample_id}: cached, skipping inference")
        continue
    t0 = time.time()
    volume, meta = load_volume(sample_id)
    out_node, out_edge = predict_one(model, volume, meta)
    np.savez_compressed(
        cache_file,
        node=np.asarray(out_node, dtype=np.float64),
        edge=np.asarray(out_edge, dtype=np.float64),
    )
    print(f"{sample_id}: nodes={len(out_node):,} candidate_edges={len(out_edge):,} "
          f"({time.time()-t0:.1f}s)", flush=True)

del model
torch.cuda.empty_cache()
print("STAGE A complete")


In [ ]:
# STAGE B - grid over ILP costs on the cached candidate graphs, official scoring.
import time
from collections import defaultdict

def count_forks(graph):
    outdeg = defaultdict(int)
    for row in graph.edge_attrs().iter_rows(named=True):
        outdeg[int(row["source_id"])] += 1
    return sum(1 for v in outdeg.values() if v >= 2)

truth = {}
for sample_id in valid_id:
    ds = open_dataset(f"{KAGGLE_DIR}/train/{sample_id}.zarr",
                      normalize=False, load_image=False, require_tracks=True)
    meta_gt = GeffMetadata.read(f"{KAGGLE_DIR}/train/{sample_id}.geff")
    truth[sample_id] = {"graph": ds.tracks, "scale": ds.scale,
                        "n_total": float(meta_gt.extra["estimated_number_of_nodes"])}

SIXBBA = [s for s in valid_id if s.startswith("6bba")]
rows = []
for app, dis, ew in COST_GRID:
    ew_eff = ILP_EDGE_WEIGHT if ew is None else ew
    tag = f"app={app} dis={dis} edge={ew_eff}"
    print(f"\n{'='*70}\n{tag}\n{'='*70}", flush=True)
    per = []
    for sample_id in valid_id:
        d = np.load(f"{CACHE_DIR}/{sample_id}.npz")
        graph = build_graph(d["node"],
            [(int(i), int(j), float(p), float(dd)) for i, j, p, dd in d["edge"]])
        t0 = time.time()
        if graph.num_edges() > 0:
            solver = td.solvers.ILPSolver(
                edge_weight=ew_eff * td.EdgeAttr("edge_prob"),
                appearance_weight=app,
                disappearance_weight=dis,
                division_weight=ILP_DIVISION_WEIGHT_FIXED,
                num_threads=1,
            )
            graph = solver.solve(graph)
            if hasattr(graph, "detach"):
                graph = graph.detach()   # evaluate() calls .copy(), unsupported on GraphView
        solve_s = time.time() - t0
        try:
            er = evaluate(graph, truth[sample_id]["graph"],
                          scale=truth[sample_id]["scale"], max_distance=7.0)
            rec = node_recall(graph, truth[sample_id]["graph"])
            m = per_sample_metrics(er=er, n_total=truth[sample_id]["n_total"], node_recall=rec)
        except Exception as exc:
            print(f"  {sample_id}: SCORING FAILED ({type(exc).__name__}: {exc})", flush=True)
            continue
        r = {"appearance": app, "disappearance": dis, "edge_weight": ew_eff,
             "dataset": sample_id, "nodes": graph.num_nodes(), "edges": graph.num_edges(),
             "forks": count_forks(graph),
             "edge_tp": er.edge_tp, "edge_fp": er.edge_fp, "edge_fn": er.edge_fn,
             "adj_edge_jaccard": m["adj_edge_jaccard"], "solve_s": round(solve_s,1)}
        per.append(r); rows.append(r)
        print(f"  {sample_id}: nodes={graph.num_nodes():,} edges={graph.num_edges():,} "
              f"eTP/FP/FN={er.edge_tp}/{er.edge_fp}/{er.edge_fn} "
              f"adjJ={m['adj_edge_jaccard']:.5f} ({solve_s:.0f}s)", flush=True)

    def agg(subset):
        D = sum(r["edge_tp"]+r["edge_fp"]+r["edge_fn"] for r in subset)
        return (sum(r["adj_edge_jaccard"]*(r["edge_tp"]+r["edge_fp"]+r["edge_fn"])
                    for r in subset)/D) if D else 0.0
    a_all = agg(per)
    a_6bba = agg([r for r in per if r["dataset"] in SIXBBA])
    print(f"\n  >>> {tag}: adjJ_all={a_all:.6f}  adjJ_6bba={a_6bba:.6f}", flush=True)
    rows.append({"appearance": app, "disappearance": dis, "edge_weight": ew_eff,
                 "dataset": "__AGGREGATE__", "adj_edge_jaccard": a_all,
                 "adj_edge_jaccard_6bba": a_6bba,
                 "nodes": sum(r["nodes"] for r in per),
                 "edges": sum(r["edges"] for r in per)})

pd.DataFrame(rows).to_csv("/kaggle/working/exp118_cost_grid.csv", index=False)
print("\nwrote /kaggle/working/exp118_cost_grid.csv")


In [ ]:
# SUMMARY - rank by the 6bba-only aggregate (the discriminating signal)
df = pd.DataFrame(rows)
agg = df[df.dataset == "__AGGREGATE__"].sort_values("adj_edge_jaccard_6bba", ascending=False)
print("Ranked by adjJ on the two 6bba movies (~94% of local weight):\n")
print(agg[["appearance","disappearance","edge_weight","nodes","edges",
           "adj_edge_jaccard","adj_edge_jaccard_6bba"]].to_string(index=False))

base = agg[(agg.appearance==0.0)&(agg.disappearance==1.4)&(agg.edge_weight==-1.0)]
best = agg.iloc[0]
print(f"\nBEST: app={best.appearance} dis={best.disappearance} edge={best.edge_weight} "
      f"-> adjJ_6bba={best.adj_edge_jaccard_6bba:.6f}")
if len(base):
    b = base.iloc[0]
    print(f"INHERITED (0.0/1.4/-1.0): adjJ_6bba={b.adj_edge_jaccard_6bba:.6f}")
    print(f"DELTA: {best.adj_edge_jaccard_6bba - b.adj_edge_jaccard_6bba:+.6f}")
    print("\nNOTE: local under-reads public LB by ~0.035; treat as RANKING only.")
